# Data Collection — Wikipedia Web Scraping
**Purpose:** collect historical Falcon 9 launch table data from Wikipedia using `requests` and BeautifulSoup.

The page structure may change. The notebook prints the tables it finds so the extraction can be checked rather than silently assuming a fixed table index.

In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display

URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
headers = {"User-Agent": "Mozilla/5.0 (compatible; educational data science project)"}
response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all("table", class_="wikitable")
print("Wikipedia tables found:", len(tables))

Wikipedia tables found: 4


In [3]:

frames = []

for idx, table in enumerate(tables):
    try:
        # Read the table's HTML using StringIO
        from io import StringIO

        html_table = StringIO(str(table))
        parsed_tables = pd.read_html(html_table)

        if parsed_tables and not parsed_tables[0].empty:
            frame = parsed_tables[0]
            frame["source_table_index"] = idx
            frames.append(frame)

    except (ValueError, ImportError) as error:
        print(f"Skipping table {idx}: {error}")
        continue

print("Tables successfully parsed:", len(frames))

if not frames:
    raise RuntimeError(
        "No tables were parsed. Check the Wikipedia response and table structure."
    )

display(frames[0].head())




Tables successfully parsed: 4


,Flight No.,Date and time (UTC),"Version, booster[i]",Launch site,Payload[j],Payload mass,Orbit,Customer,Launch outcome,Booster landing,source_table_index
0,418,"January 4, 2025 01:27[29]",F9 B5 B1073‑20,"Cape Canaveral, SLC‑40",Thuraya 4-NGS,"5,000 kg (11,000 lb)",GTO,Thuraya,Success,Success (ASOG),0
1,418,Planned replacement for Thuraya 2 and 3.[30][31],Planned replacement for Thuraya 2 and 3.[30][31],Planned replacement for Thuraya 2 and 3.[30][31],Planned replacement for Thuraya 2 and 3.[30][31],Planned replacement for Thuraya 2 and 3.[30][31],Planned replacement for Thuraya 2 and 3.[30][31],Planned replacement for Thuraya 2 and 3.[30][31],Planned replacement for Thuraya 2 and 3.[30][31],Planned replacement for Thuraya 2 and 3.[30][31],0
2,419,"January 6, 2025 20:43[32]",F9 B5 B1077‑17,"Cape Canaveral, SLC‑40",Starlink: Group 6‑71,"~17,500 kg (38,600 lb)",LEO,SpaceX,Success,Success (JRTI),0
3,419,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,0
4,420,"January 8, 2025 15:27[33]",F9 B5 B1086‑3,"Kennedy, LC‑39A",Starlink: Group 12-11 (21 satellites),"~16,500 kg (36,400 lb)",LEO,SpaceX,Success,Success (ASOG),0


In [ ]:
# Combine parsed tables with a source index; columns can differ across historical tables.
scraped_df = pd.concat(frames, ignore_index=True, sort=False)
print("Combined shape:", scraped_df.shape)
display(scraped_df.head())
scraped_df.to_csv("wikipedia_falcon9_launch_tables.csv", index=False)
print("Saved: wikipedia_falcon9_launch_tables.csv")

## Interpretation
Web scraping converts HTML tables into a tabular dataset for further inspection. Before using this file for modelling, standardize column names, inspect duplicate/header rows, and verify that outcome labels are consistent. The scraped tables are supporting historical data; do not assume they are directly interchangeable with the course's modelling dataset.